# DML IRM vs Propensity Score Weighting (PSW)

Causal Inference consists of two main parts: Identification Assumptions and Model Specification. SUTVA, Unconfoundedness, Overlap are strong assumptions that must be true to call our inference causal. Studies and quasi experiments often have problems with Identification Assumptions so in practice you spend time to prove them, not model specification

However, in this notebook I will focus on model specification. Propensity Score Weighting is a classical baseline for observational studies, estimating ATTE from an estimated propensity model. It should perform worse than the DML IRM approach because:

- uses only the propensity model and does not learn a separate outcome regression
- is more sensitive to extreme or misscaled propensities because errors become large weights
- does not use orthogonalization, so nuisance-model misspecification shows up directly in the estimate
- does not use cross-fitting to control overfitting bias from flexible ML models
- often has much worse effective sample size than the nominal row count when overlap is weak
- provides a weaker default benchmark than a doubly robust IRM when both treatment and outcome are nonlinear

We will compare absolute estimates on DGPs from Causalis between the IRM DML model implemented in Causalis and a hand-written PSW estimator.

In [1]:
import time
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, CatBoostRegressor

from causalis.data_contracts import CausalData
from causalis.scenarios.unconfoundedness import IRM
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_26_rich
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_binary_26

# generate_obs_hte_26_rich()

Read more about dgp at https://causalis.causalcraft.com/articles/generate_obs_hte_26_rich

In [2]:
ORACLE_COLS = {"m", "m_obs", "tau_link", "g0", "g1", "cate"}
TREATMENT_COL = "d"
OUTCOME_COL = "y"
SIZES = [10_000, 100_000, 1_000_000]
SEED = 42

In [3]:
def infer_confounders(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in ORACLE_COLS.union({TREATMENT_COL, OUTCOME_COL})]

In [4]:
def estimate_irm_atte(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float]:
    t0 = time.perf_counter()
    cd = CausalData(
        df=df[[OUTCOME_COL, TREATMENT_COL] + confounders].copy(),
        treatment=TREATMENT_COL,
        outcome=OUTCOME_COL,
        confounders=confounders,
    )

    irm = IRM(
        n_jobs=-1,
        random_state=seed,
        ml_g=CatBoostRegressor(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
        ml_m=CatBoostClassifier(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
        store_diagnostics=False
    )

    atte = float(irm.fit(cd).estimate(score="ATTE").value)
    return atte, (time.perf_counter() - t0)

In [5]:
def _effective_sample_size(weights: np.ndarray) -> float:
    weights = np.asarray(weights, dtype=float).ravel()
    total = float(weights.sum())
    sq_total = float(np.square(weights).sum())
    if total <= 0.0 or sq_total <= 0.0:
        return 0.0
    return float(total ** 2 / sq_total)

def estimate_psw_atte_total(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float, float, float]:
    t0 = time.perf_counter()

    ps_model = CatBoostClassifier(
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
        random_seed=seed,
    )
    ps_model.fit(df[confounders], df[TREATMENT_COL].astype(int), verbose=False)

    propensity = np.clip(
        ps_model.predict_proba(df[confounders])[:, 1],
        1e-2,
        1 - 1e-2,
    )

    treated_mask = df[TREATMENT_COL].to_numpy(dtype=bool)
    control_mask = ~treated_mask
    control_weights = propensity[control_mask] / (1.0 - propensity[control_mask])

    treated_mean = float(df.loc[treated_mask, OUTCOME_COL].mean())
    control_mean = float(np.average(df.loc[control_mask, OUTCOME_COL], weights=control_weights))
    atte = treated_mean - control_mean

    return (
        float(atte),
        (time.perf_counter() - t0),
        _effective_sample_size(control_weights),
        float(control_weights.max(initial=0.0)),
    )

In [6]:
results = []

for n in SIZES:
    print(f"Running n={n:,} ...")

    df = generate_obs_hte_26_rich(
        n=n,
        seed=SEED,
        include_oracle=True,
        return_causal_data=False,
    )
    confounders = infer_confounders(df)

    ground_truth_atte = float(df.loc[df[TREATMENT_COL] == 1, "cate"].mean())
    irm_atte, irm_runtime_sec = estimate_irm_atte(df=df, confounders=confounders, seed=SEED)

    psw_atte, psw_runtime_sec, psw_control_ess, psw_max_weight = estimate_psw_atte_total(
        df=df,
        confounders=confounders,
        seed=SEED,
    )

    results.append(
        {
            "n": n,
            "ground_truth_atte": ground_truth_atte,
            "irm_atte": irm_atte,
            "psw_atte": psw_atte,
            "irm_abs_error": abs(irm_atte - ground_truth_atte),
            "psw_abs_error": abs(psw_atte - ground_truth_atte),
            "irm_runtime_sec": irm_runtime_sec,
            "psw_runtime_sec": psw_runtime_sec,
            "psw_control_ess": psw_control_ess,
            "psw_max_weight": psw_max_weight,
        }
    )

comparison = pd.DataFrame(results)
comparison

Running n=10,000 ...
Running n=100,000 ...
Running n=1,000,000 ...


,n,ground_truth_atte,irm_atte,psw_atte,irm_abs_error,psw_abs_error,irm_runtime_sec,psw_runtime_sec,psw_control_ess,psw_max_weight
0,10000,11.454404,6.256087,5.451640,5.198317,6.002764,7.732994,3.222296,5339.475134,0.388524
1,100000,10.914991,12.106856,11.189699,1.191864,0.274708,47.719494,8.852210,61532.484911,0.546033
2,1000000,11.028129,10.340542,10.062838,0.687587,0.965291,339.667114,36.398603,622717.546229,1.203364


In [7]:
for _, row in comparison.iterrows():
    n = int(row["n"])
    print(
        f"n={n:,}: ground truth ATTE={row['ground_truth_atte']:.6f}, "
        f"IRM ATTE={row['irm_atte']:.6f}, PSW ATTE={row['psw_atte']:.6f}, "
        f"PSW control ESS={row['psw_control_ess']:.1f}"
    )

n=10,000: ground truth ATTE=11.454404, IRM ATTE=6.256087, PSW ATTE=5.451640, PSW control ESS=5339.5
n=100,000: ground truth ATTE=10.914991, IRM ATTE=12.106856, PSW ATTE=11.189699, PSW control ESS=61532.5
n=1,000,000: ground truth ATTE=11.028129, IRM ATTE=10.340542, PSW ATTE=10.062838, PSW control ESS=622717.5


DML IRM should be more stable than PSW on the rich nonlinear DGP, especially when PSW loses effective sample size through large ATT weights.

# generate_obs_hte_binary_26()

Read more about the dgp at https://causalis.causalcraft.com/articles/generate_obs_hte_binary_26

In [8]:
ORACLE_COLS = {"m", "m_obs", "tau_link", "g0", "g1", "cate"}
TREATMENT_COL = "d"
OUTCOME_COL = "y"
SIZES = [10_000, 100_000, 1_000_000]
SEED = 42

In [9]:
def infer_confounders(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in ORACLE_COLS.union({TREATMENT_COL, OUTCOME_COL})]

In [10]:
def estimate_irm_atte(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float]:
    t0 = time.perf_counter()
    cd = CausalData(
        df=df[[OUTCOME_COL, TREATMENT_COL] + confounders].copy(),
        treatment=TREATMENT_COL,
        outcome=OUTCOME_COL,
        confounders=confounders,
    )

    irm = IRM(
        n_jobs=-1,
        random_state=seed,
        ml_g=CatBoostRegressor(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
        ml_m=CatBoostClassifier(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
        store_diagnostics=False
    )

    atte = float(irm.fit(cd).estimate(score="ATTE").value)
    return atte, (time.perf_counter() - t0)

In [11]:
def _effective_sample_size(weights: np.ndarray) -> float:
    weights = np.asarray(weights, dtype=float).ravel()
    total = float(weights.sum())
    sq_total = float(np.square(weights).sum())
    if total <= 0.0 or sq_total <= 0.0:
        return 0.0
    return float(total ** 2 / sq_total)

def estimate_psw_atte_total(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float, float, float]:
    t0 = time.perf_counter()

    ps_model = CatBoostClassifier(
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
        random_seed=seed,
    )
    ps_model.fit(df[confounders], df[TREATMENT_COL].astype(int), verbose=False)

    propensity = np.clip(
        ps_model.predict_proba(df[confounders])[:, 1],
        1e-2,
        1 - 1e-2,
    )

    treated_mask = df[TREATMENT_COL].to_numpy(dtype=bool)
    control_mask = ~treated_mask
    control_weights = propensity[control_mask] / (1.0 - propensity[control_mask])

    treated_mean = float(df.loc[treated_mask, OUTCOME_COL].mean())
    control_mean = float(np.average(df.loc[control_mask, OUTCOME_COL], weights=control_weights))
    atte = treated_mean - control_mean

    return (
        float(atte),
        (time.perf_counter() - t0),
        _effective_sample_size(control_weights),
        float(control_weights.max(initial=0.0)),
    )

In [12]:
results = []

for n in SIZES:
    print(f"Running n={n:,} ...")

    df = generate_obs_hte_binary_26(
        n=n,
        seed=SEED,
        include_oracle=True,
        return_causal_data=False,
    )
    confounders = infer_confounders(df)

    ground_truth_atte = float(df.loc[df[TREATMENT_COL] == 1, "cate"].mean())
    irm_atte, irm_runtime_sec = estimate_irm_atte(df=df, confounders=confounders, seed=SEED)

    psw_atte, psw_runtime_sec, psw_control_ess, psw_max_weight = estimate_psw_atte_total(
        df=df,
        confounders=confounders,
        seed=SEED,
    )

    results.append(
        {
            "n": n,
            "ground_truth_atte": ground_truth_atte,
            "irm_atte": irm_atte,
            "psw_atte": psw_atte,
            "irm_abs_error": abs(irm_atte - ground_truth_atte),
            "psw_abs_error": abs(psw_atte - ground_truth_atte),
            "irm_runtime_sec": irm_runtime_sec,
            "psw_runtime_sec": psw_runtime_sec,
            "psw_control_ess": psw_control_ess,
            "psw_max_weight": psw_max_weight,
        }
    )

comparison = pd.DataFrame(results)
comparison

Running n=10,000 ...
Running n=100,000 ...
Running n=1,000,000 ...


,n,ground_truth_atte,irm_atte,psw_atte,irm_abs_error,psw_abs_error,irm_runtime_sec,psw_runtime_sec,psw_control_ess,psw_max_weight
0,10000,0.103885,0.102344,0.113561,0.001541,0.009676,4.270595,1.746638,5777.856137,0.838468
1,100000,0.101238,0.103547,0.106411,0.002309,0.005173,24.387916,4.752561,61172.870093,1.230113
2,1000000,0.101411,0.103282,0.104150,0.001871,0.002739,215.242940,33.692571,623116.282826,1.637482


In [13]:
for _, row in comparison.iterrows():
    n = int(row["n"])
    print(
        f"n={n:,}: ground truth ATTE={row['ground_truth_atte']:.6f}, "
        f"IRM ATTE={row['irm_atte']:.6f}, PSW ATTE={row['psw_atte']:.6f}, "
        f"PSW control ESS={row['psw_control_ess']:.1f}"
    )

n=10,000: ground truth ATTE=0.103885, IRM ATTE=0.102344, PSW ATTE=0.113561, PSW control ESS=5777.9
n=100,000: ground truth ATTE=0.101238, IRM ATTE=0.103547, PSW ATTE=0.106411, PSW control ESS=61172.9
n=1,000,000: ground truth ATTE=0.101411, IRM ATTE=0.103282, PSW ATTE=0.104150, PSW control ESS=623116.3


# Conclusion

I recommend using DML IRM as the default model specification for the unconfoundedness scenario, and treating PSW as a simple benchmark rather than the default estimator.